# 15 · DeBERTa-v3-base ce4, seeds 43–46, with saved checkpoints (supplement T3)

**Why:** (1) none of the DeBERTa runs saved its weights, so the negation probe (notebook 16) and the easy-query holdout (notebook 17) need a DeBERTa checkpoint (`s43` is the one they use); (2) more seeds of the base ce4 configuration: ce4 vs MSE on DeBERTa gets more than one seed, and the **seed noise of the final system's backbone** is measured with five seeds (42 from notebook 06 plus 43–46) instead of being borrowed from MiniLM.

**Configuration** = `d2_deberta_main_ce4_s42` (notebooks 06 / 07): `microsoft/deberta-v3-base`, `main`, `max_len` 256, ce4, lr 2e-5, 1,200 warm-up steps, 1 epoch, batch 32, fp16; **seeds 43, 44, 45, 46**; random pair batches as in the original run (`sampler` not set). The runs are independent and run in parallel, one per GPU (`src/gpu_jobs.py`). Never more seeds than GPUs: one run takes about 2.75 h on a T4, so with fewer GPUs the notebook keeps the first seeds (one GPU: seed 43 only) and stays one round long.

**Checks:** each val nDCG within 0.8891 ± 0.003 (seed 42); paired differences with `d2_deberta_main_ce4_s42` (and `d1_deberta_main_mse_s43` for seed 43) if their val scores are in the inputs. Each checkpoint is saved in fp16 after predicting and reloaded once to verify that it reproduces the val scores.

**Kaggle settings:** GPU (every visible GPU is used: one run per GPU at a time). **Internet on**, or attach the output of `00_hf_models`.

**Inputs:**
- Dataset `wm-src` built from the **`supplement`** branch (`src/gpu_jobs.py`). Required.
- Optional, for the paired comparisons and the five-seed table: the notebook outputs holding `scores/d2_deberta_main_ce4_s42_val.parquet` (**06_d2_losses**) and `scores/d1_deberta_main_mse_s43_val.parquet` (**05_d1_input_backbone**). Without them those comparisons are skipped.
- Add Input → Notebook Output → **01_preprocess**. Required.
- Optional: Notebook Output → **00_hf_models** (session without Internet).

**How to run.** `MODE = 'smoke'` (5 % of train, seed 43 only, val only, checkpoint save and reload checked; prints the time of one full run), then `MODE = 'full'` → Save & Run All. Download `results/` into `data/results/supplement/`; the notebook output (`models/d2_deberta_main_ce4_s4*/`) is the checkpoint input of notebooks 16 and 17.

**Output** (`/kaggle/working/`): `results/d2_deberta_main_ce4_s{43..46}.json`, `results/scores/d2_deberta_main_ce4_s{43..46}_{val,test}.parquet` (with `p_E..p_I`), `results/deberta_base_seeds.csv`, `models/d2_deberta_main_ce4_s{43..46}/`, `jobs/<run>/log.txt`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, json, os, time

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; before torch

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.data import load_pairs
from src.evaluation import RunLog, calibration, paired_bootstrap
from src.gpu_jobs import cleanup, prefetch, run_jobs, save_frame

transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': 5% of train, seed 43, val only | 'full'

CFG = {'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 256, 'loss': 'ce4',
       'lr': 2e-5, 'warmup_steps': 1200, 'epochs': 1, 'batch_size': 32, 'weight_decay': 0.01,
       'max_grad_norm': 1.0, 'fp16': True, 'eval_batch_size': 64}      # = d2_deberta_main_ce4_s42
N_GPU = torch.cuda.device_count()
# one DeBERTa-base run takes ~2.75 h on a T4 (d2_deberta_main_ce4_s42: 143 min train + 22 min predict): never more
# seeds than GPUs, so the notebook is one round of runs and stays far from Kaggle's 12 h session limit
SEEDS = [43] if MODE == 'smoke' else [43, 44, 45, 46][:max(1, N_GPU)]
REF = 'd2_deberta_main_ce4_s42'                                    # seed 42 of the same configuration (notebook 06)
MSE_REF = {43: 'd1_deberta_main_mse_s43'}                          # MSE run of the same seed, where one exists
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']     # test is predicted and saved, never scored here
EVAL_EVERY = 300 if MODE == 'smoke' else None
PREFIX = 'smoke_' if MODE == 'smoke' else ''
NAMES = {s: f'{PREFIX}d2_deberta_main_ce4_s{s}' for s in SEEDS}
RES, MODELS, WORK = Path('/kaggle/working/results'), Path('/kaggle/working/models'), Path('/kaggle/working/jobs')
DATA = Path('/tmp/wm_jobs_data')                                  # temporary data copies (not in the output)

assert MODE in ('smoke', 'full')
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', N_GPU, 'x', torch.cuda.get_device_name(0))
print('MODE', MODE, '| runs', list(NAMES.values()))

In [ ]:
pairs = load_pairs(CFG['fields'])
train_df = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_df['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=42)
    train_df = train_df[train_df['query_id'].isin(keep)]
train_df = train_df.reset_index(drop=True)
parts = {s: pairs[pairs['split'] == s].reset_index(drop=True) for s in EVAL_SPLITS}
print(f'train {train_df["query_id"].nunique():,} queries / {len(train_df):,} pairs')

## Training runs (parallel) and checkpoints

One job per seed (`src/gpu_jobs.py`, one GPU each): train, predict val / test with `p_E..p_I`, save the checkpoint in fp16 after predicting, reload it and re-score 2,000 val pairs (the largest score difference must be within fp16 tolerance). Then every run is logged here: paired bootstrap against seed 42, test saved blind.

In [ ]:
prefetch(CFG['model'])
data = {'train': save_frame(train_df, DATA / 'train.parquet'),
        **{s: save_frame(parts[s], DATA / f'{s}.parquet') for s in EVAL_SPLITS}}
jobs = [{'name': NAMES[s], 'kind': 'train', 'cfg': CFG, 'seed': s, 'train': data['train'],
         'predict': {sp: data[sp] for sp in EVAL_SPLITS}, 'eval_every': EVAL_EVERY,
         'print_every': 100 if MODE == 'smoke' else 1000, 'save_dir': str(MODELS / NAMES[s]), 'reload_check': 2000}
        for s in SEEDS]
results = run_jobs(jobs, WORK, N_GPU)

log = RunLog(RES, {s: parts[s][['example_id', 'query_id', 'esci_label']] for s in EVAL_SPLITS})
has_ref = MODE == 'full' and log.load(REF, splits=['val'])
recs = {}
for s in SEEDS:
    name, r = NAMES[s], results[NAMES[s]]
    recs[s] = log.log(name, r['scores'],
                      {'mode': MODE, **CFG, 'seed': s, 'subsample': SUBSAMPLE,
                       'train_queries': int(train_df['query_id'].nunique()), 'train_pairs': len(train_df),
                       'n_gpus': 1, 'parallel_jobs': N_GPU, **r['info']},
                      compare_to=REF if has_ref else None, blind=('test',), extra=r['probs'])
    if MODE == 'full' and s in MSE_REF and log.load(MSE_REF[s], splits=['val']):
        print(f'val {name} - {MSE_REF[s]}:', paired_bootstrap(log.per_query[MSE_REF[s]]['val'], log.per_query[name]['val']))
    print(f'{name} calibration (val):', calibration(r['probs']['val'].to_numpy(), parts['val']['esci_label'].to_numpy()))
    (MODELS / name / 'checkpoint_info.json').write_text(json.dumps(
        {'run': name, 'cfg': CFG, 'seed': s, 'val_ndcg': recs[s]['metrics']['val']['ndcg'], 'src_version': src_version()},
        indent=1))
    gap = r['info']['reload_gap']
    print(f"{name}: checkpoint {r['info']['checkpoint_mb']} MB | max |score difference| after reload {gap:.4f}")
    assert gap < 0.02, f'{name}: the reloaded checkpoint does not reproduce the scores'
cleanup(WORK, DATA)

## Checks and seed noise

Smoke: loss goes down, val above random, time of one full run. Full: each seed within 0.8891 ± 0.003, and the spread of val nDCG over the five seeds of this configuration (42 from notebook 06 if its scores are attached, 43–46 here): standard deviation over seeds and the range. This is the seed noise of one DeBERTa-base cross-encoder (the final system averages fold models, so its own noise is smaller).

In [ ]:
if MODE == 'smoke':
    rec = recs[SEEDS[0]]
    v = rec['metrics']['val']['ndcg']
    first, last = rec['loss_log'][0]['loss'], rec['loss_log'][-1]['loss']
    ok = last < first and v > 0.742
    print(f"{'OK   ' if ok else 'CHECK'} loss {first:.4f} -> {last:.4f} | val nDCG {v} | "
          f"full epoch ~{rec['steps'] / SUBSAMPLE / rec['it_per_s'] / 3600:.2f} h")
    print('Smoke test passed: switch MODE to full.' if ok else 'Training did not learn: send me this output.')
else:
    for s in SEEDS:
        v = recs[s]['metrics']['val']['ndcg']
        print(f"{'OK   ' if abs(v - 0.8891) <= 0.003 else 'CHECK'} seed {s}: val nDCG {v} (seed 42: 0.8891; expected within 0.003)")
    names = ([REF] if has_ref else []) + [NAMES[s] for s in SEEDS]
    seeds = pd.DataFrame([{'run': n, 'val_ndcg': round(float(log.per_query[n]['val']['ndcg'].mean()), 4)} for n in names])
    display(seeds)
    sd, rng = seeds['val_ndcg'].std(ddof=1), seeds['val_ndcg'].max() - seeds['val_ndcg'].min()
    print(f'{len(seeds)} seeds: std {sd:.4f} | range {rng:.4f} (MiniLM seed noise used so far: 0.0014)')
    seeds.assign(std_over_seeds=round(float(sd), 4)).to_csv(RES / 'deberta_base_seeds.csv', index=False)